# AMD-Latent
### Training AMD classifiers

---

The current notebook train and validate AMD score Machine Learning classifiers on extracted features.

# Packages import and functions defining

Loading the required packages and definition of functions.

In [ ]:
# PACKAGES IMPORT

# System packages
import os
import numpy as np
from pathlib import Path
import pandas as pd

# Jupyter packages
import ipywidgets as widgets
from amdlatent.utils.widgets import filechooser, toggleoptions, rangeslideroptions, slideroptions, textoptions

# Classification packages
from amdlatent.dataset.extraction import setsextraction, besthypersextraction
from amdlatent.models.evaluation import evaluatemodel, predextr
from amdlatent.models.classifiers import trainmodel, fitseed
import pickle as pkl

# Statistics packages
from tqdm import tqdm

# Path checker
if Path().resolve().name != "AMD-Latent": os.chdir(os.path.join(Path().resolve().parents[1]))

# Code execution

Select the following options for running the script and execute it.

In [ ]:
# OPTIONS SELECTION

from amdlatent.utils.widgets import textoptions


dirselect = filechooser("Select the <b>AMD Study</b> folder:")
clipselect = toggleoptions(['RETCLIP', 'CLIP', 'ResNet50'], "Select the <b>model</b> to use:")
modeselect = toggleoptions(['Training', 'Robustness'], "Select the <b>execution</b> approach:")

splitselect = toggleoptions(["Couples-wise", "Patient-wise", "Couples stratified-wise"],"Select the <b>dataset split type</b>:")
classifierselect = toggleoptions(['Random Forest', 'AdaBoost', 'SVM', 'SVM Linear', 'MLP'], "Select the <b>classifier</b> to use:")
repeatselect = slideroptions("Select the <b>number of re-training</b> to try with the best hyperparameters:", 0, 10, 1, 3)

hyperbox = widgets.Output()
display(hyperbox)
hyperselect = None
epochselect = None
metricbox = widgets.Output()
display(metricbox)
metricselect = None

def _update_hyperselect(change=None):
    global hyperselect
    global epochselect
    global metricselect
    global promptstype
    hyperbox.clear_output()
    metricbox.clear_output()
    if modeselect.value == "Training":
        with metricbox:
            metricselect = toggleoptions(['Accuracy', 'Balanced Accuracy'], "Select the <b>metric</b> of the <b>validation set</b> on which best selection is based:")
        if classifierselect.value == "Random Forest" or classifierselect.value == "AdaBoost":
            with hyperbox:
                hyperselect = rangeslideroptions("Select the <b>range of estimators</b> to try:", 100, 1000, 10, (100, 400))
                epochselect = None
        elif classifierselect.value == "SVM" or classifierselect.value == "SVM Linear":
            with hyperbox:
                hyperselect = rangeslideroptions("Select the <b>10^x range of C and/or gamma</b> parameters to try:", -4, 4, 1, (-3, 2))
                epochselect = None
        elif classifierselect.value == "MLP":
            with hyperbox:
                hyperselect = rangeslideroptions("Select the <b>range of hidden layers</b> numbers to try:", 2, 8, 1, (2, 5))
                epochselect = None
        else: 
            hyperselect = None
            epochselect = None
    else: 
        hyperselect = None
        epochselect = None
        metricselect = None

def _bind_hyperselect(change=None):
    if classifierselect is not None:
        classifierselect.observe(_update_hyperselect, names="value")
    if modeselect is not None:
        modeselect.observe(_update_hyperselect, names="value")
    _update_hyperselect()
_bind_hyperselect()

classselect = toggleoptions(['All', 'Binary No', 'Binary Late'], "Select the <b>classification type</b> to use:")

In [ ]:
# SCRIPT EXECUTION

metadata = {
    "directory": dirselect.selected_path,
    "clip": clipselect.value,
    "mode": modeselect.value,
    "splittype": splitselect.value,
    "classifier": classifierselect.value,
    "repeat": repeatselect.value,
    "classtype": classselect.value,
    "metric": metricselect.value if metricselect is not None else None,
    "hyper": hyperselect.value if hyperselect is not None else None,
    "epoch": epochselect.value if epochselect is not None else None,
}

dataset = pd.read_excel(os.path.join(metadata["directory"], "Data", "DatasetComplete.xlsx"), sheet_name='AMD')
dataset.columns = dataset.columns.astype(str).str.strip()
X_train, y_train, X_val, y_val, X_test, y_test, trainpaths = setsextraction(dataset, metadata["classtype"], metadata["splittype"], encoder = metadata["clip"].lower())

if metadata["classtype"] == "Binary No" or metadata["classtype"] == "Binary Late":
    pdheader = [("Classifier", "", ""), ("Hyperparameters", "", ""), ("Accuracy", "Train", ""), ("Accuracy", "Val", ""), ("Accuracy", "Test", ""), ("Balanced Accuracy", "Train", ""), ("Balanced Accuracy", "Val", ""), ("Balanced Accuracy", "Test", ""),("Sensitivity", "Train", ""), ("Sensitivity", "Val", ""), ("Sensitivity", "Test", ""), ("Specificity", "Train", ""), ("Specificity", "Val", ""), ("Specificity", "Test", ""), ("PPV", "Train", ""), ("PPV", "Val", ""), ("PPV", "Test", ""), ("NPV", "Train", ""), ("NPV", "Val", ""), ("NPV", "Test", ""), ("ROC AUC", "Train", ""), ("ROC AUC", "Val", ""), ("ROC AUC", "Test", "")]
else:
    pdheader = [("Classifier", "", ""), ("Hyperparameters", "", ""), ("Accuracy", "Train", ""), ("Accuracy", "Val", ""), ("Accuracy", "Test ", ""), ("Balanced Accuracy ", "Train ", ""), ("Balanced Accuracy ", "Val ", ""), ("Balanced Accuracy ", "Test ", ""), ("MAE", "Train", ""), ("MAE", "Val", ""), ("MAE", "Test", "")]
for subset in ["Train", "Val", "Test"]:
    for nclass in range(len(np.unique(y_train))):
        pdheader.append(("Class Accuracy", subset, f"Class {nclass}"))

pdcolumns = pd.MultiIndex.from_tuples(pdheader, names=["Value", "Set", "Class"])
classifierinfo = pd.DataFrame(columns=pdcolumns)

if metadata["mode"] == "Training":
    if metadata["classifier"] == 'Random Forest': hypers = range(metadata["hyper"][0], metadata["hyper"][1]+1, 10)
    elif metadata["classifier"] == 'SVM': hypers = [(c, g) for c in [10**x for x in range(metadata["hyper"][0], metadata["hyper"][1]+1)] for g in [10**x for x in range(metadata["hyper"][0], metadata["hyper"][1]+1)]]
    elif metadata["classifier"] == 'SVM Linear': hypers = [10**x for x in range(metadata["hyper"][0], metadata["hyper"][1]+1)]
    elif metadata["classifier"] == 'AdaBoost': hypers = range(metadata["hyper"][0], metadata["hyper"][1]+1, 10)
    elif metadata["classifier"] == 'MLP': hypers = [tuple(2**(j+4) for j in range(i)) for i in range(metadata["hyper"][0], metadata["hyper"][1] + 1)]
    pbar = tqdm(hypers, desc="Training classifiers " + metadata["classifier"])
    for hyper in pbar:
        model = trainmodel(X_train, y_train, metadata["classifier"], hyper, randomstate=fitseed(metadata["classtype"], metadata["classifier"], metadata["splittype"], hyper))
        currinfo = evaluatemodel(model, X_train, y_train, X_val, y_val, X_test, y_test, hyper, metadata["classifier"])
        currinfo = pd.DataFrame([currinfo], columns=classifierinfo.columns)
        bestmetric = classifierinfo[metadata["metric"]]["Val"].max() if not classifierinfo.empty else -np.inf
        if currinfo.loc[0, (metadata["metric"], "Val", "")] > bestmetric:
            bestmodel = model
            besthyper = hyper
            pbar.set_postfix_str(f"Best hyperparameters: {currinfo.loc[0, 'Hyperparameters']}, {metadata['metric']}: {currinfo.loc[0, (metadata['metric'], 'Val', '')]:.3f}")
        classifierinfo = pd.concat([classifierinfo, currinfo], ignore_index=True)

if metadata["mode"] == "Robustness":
    besthyper = besthypersextraction(metadata["splittype"], metadata["classtype"], metadata["classifier"], encoder = metadata["clip"].lower())

test_preds_list = []
val_preds_list = []
val_true_list = []

rpbar = tqdm(range(int(metadata["repeat"])), desc="Re-training best model " + metadata["classifier"])
for i in rpbar:
    if metadata["mode"] == "Robustness":
        X_train, y_train, X_val, y_val, _, _, _ = setsextraction(dataset, metadata["classtype"], metadata["splittype"], mixing = True, randomstate = i, encoder = metadata["clip"].lower())
    model = trainmodel(X_train, y_train, metadata["classifier"], besthyper, randomstate=fitseed(metadata["classtype"], metadata["classifier"], metadata["splittype"], i))
    currinfo = evaluatemodel(model, X_train, y_train, X_val, y_val, X_test, y_test, besthyper, metadata["classifier"])
    currinfo = pd.DataFrame([currinfo], columns=classifierinfo.columns)
    if metadata["mode"] == "Training" and currinfo.loc[0, (metadata["metric"], "Val", "")] > bestmetric:
        bestmodel  = model
        rpbar.set_postfix_str(f"Best hyperparameters: {currinfo.loc[0, 'Hyperparameters']}, {metadata['metric']}: {currinfo.loc[0, (metadata['metric'], 'Val', '')]:.3f}")
    if metadata["mode"] == "Robustness":
        test_preds_list.append(np.asarray(predextr(model, X_test, metadata["classifier"])))
        val_preds_list.append(np.asarray(predextr(model, X_val, metadata["classifier"])))
        val_true_list.append(np.asarray(y_val))
    classifierinfo = pd.concat([classifierinfo, currinfo], ignore_index=True)

if metadata["mode"] == "Training":
    if not os.path.exists(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "models", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''))): os.makedirs(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "models", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', '')))
    with open(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "models", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''), f"{metadata['classifier'].lower().replace(' ', '_')}.pkl"), 'wb') as f: pkl.dump(bestmodel, f)
    if not os.path.exists(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "metrics", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''))): os.makedirs(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "metrics", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', '')))
    classifierinfo.to_csv(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "metrics", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''), f"{metadata['classifier'].lower().replace(' ', '_')}.csv"), index=False)
elif metadata["mode"] == "Robustness":
    if not os.path.exists(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "robustness", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''))): os.makedirs(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "robustness", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', '')))
    classifierinfo.to_csv(os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "robustness", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''), f"{metadata['classifier'].lower().replace(' ', '_')}.csv"), index=False)

    predictionsdir = os.path.join("results", "areds", "classifiers", metadata['clip'].lower(), "predictions", metadata["splittype"].split('-')[0].lower().replace(' ', ''), metadata["classtype"].lower().replace(' ', ''))
    if not os.path.exists(predictionsdir): os.makedirs(predictionsdir)
    predictions = {
        "classifier": metadata["classifier"],
        "besthyper": besthyper,
        "repeats": len(test_preds_list),
        "test_true": np.asarray(y_test),
        "test_preds": np.vstack(test_preds_list),
        "val_true": val_true_list,
        "val_preds": val_preds_list,
    }
    with open(os.path.join(predictionsdir, f"{metadata['classifier'].lower().replace(' ', '_')}.pkl"), 'wb') as f: pkl.dump(predictions, f)
with pd.option_context('display.max_columns', None): display(classifierinfo)